#### Zero-shot baseline and answer-only loss

In [1]:
import lab06_starter as S
import torch

In [2]:
def answer_only_labels(input_ids, attention_mask, prompt_lens):
    labels = torch.full_like(input_ids, -100)
    for i, p_len in enumerate(prompt_lens):
        # The answer token is immediately after the prompt
        answer_idx = p_len.item()
        if answer_idx < input_ids.shape[1]:
            labels[i, answer_idx] = input_ids[i, answer_idx]
    return labels

def run_piece1():
    print("--- Zero-Shot Baseline & Answer-Only Loss ---")
    model, tok = S.load_model()
    train_ex, test_ex, label_ids = S.load_task("emotion", n_train=600, n_test=500, tokenizer=tok)

    # 1. Zero-shot evaluation
    zs_acc = S.evaluate(model, tok, test_ex, label_ids)
    print(f"Zero-shot accuracy: {zs_acc:.3f} (Chance: {1/6:.3f})")

    # 2. Test answer_only_labels assertions
    batch = train_ex[:4]
    input_ids, mask, plens = S.pad_batch(batch, tok.pad_token_id)
    labels = answer_only_labels(input_ids, mask, plens)

    # Assertions to verify correctness
    for i in range(len(batch)):
        scored_positions = (labels[i] != -100).nonzero(as_tuple=True)[0]
        # one position scored
        assert len(scored_positions) == 1, f"Example {i} scored {len(scored_positions)} positions instead of 1."
        # That position must be the answer token
        assert scored_positions[0].item() == plens[i].item(), f"Scored position {scored_positions[0].item()} does not match prompt length {plens[i].item()}."
        # No prompt token is scored
        assert scored_positions[0].item() >= plens[i].item()

    print("Assertions passed: answer_only_labels correctly targets exactly one answer token per example.")
    print("\nExplanation:")
    print("1. What starter's version trains the model to do with the prompt?")
    print("all_tokens_labels scores every single token in both the prompt and the answer. This trains the model on language modeling over the prompt text itself, treating prompt words as targets.")
    print("\n2. Why that is not the skill you are measuring?")
    print("We want to measure classification accuracy based purely on the model's ability to assign the highest probability to the correct label token at the cue position, rather than wasting gradient updates on predicting the input prompt syntax.")

if __name__ == "__main__":
    run_piece1()

--- Zero-Shot Baseline & Answer-Only Loss ---


config.json:   0%|          | 0.00/704 [00:00<?, ?B/s]

C:\Users\harma\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\harma\.cache\huggingface\hub\models--HuggingFaceTB--SmolLM2-135M. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/3.66k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

... (15 outputs left out)

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Zero-shot accuracy: 0.326 (Chance: 0.167)
Assertions passed: answer_only_labels correctly targets exactly one answer token per example.

Explanation:
1. What starter's version trains the model to do with the prompt?
all_tokens_labels scores every single token in both the prompt and the answer. This trains the model on language modeling over the prompt text itself, treating prompt words as targets.

2. Why that is not the skill you are measuring?
We want to measure classification accuracy based purely on the model's ability to assign the highest probability to the correct label token at the cue position, rather than wasting gradient updates on predicting the input prompt syntax.
